In [ ]:
# Cell 1 - run information (required)
import os, subprocess, sys, random, numpy as np, torch
# Locate the repo: run from a local clone, or clone the tagged commit on Kaggle/Colab.
REPO_URL = os.environ.get("REPO_URL", "https://github.com/shulamite512/DATA266_LAB2_PAIR44.git")  # EDIT
GIT_REF = os.environ.get("GIT_REF", "p1-v1.0")
if os.path.isdir("../../src"):  # notebooks/<member>/
    os.chdir("../..")
elif os.path.isdir("../src"):
    os.chdir("..")
import glob, shutil
zips = sorted(glob.glob("/content/**/L2_colab*.zip", recursive=True)
              + glob.glob("/kaggle/input/**/L2_colab*.zip", recursive=True), key=os.path.getmtime)
if not os.path.isdir("src") and zips:  # uploaded project zip (newest one wins)
    dest = "/content" if zips[-1].startswith("/content") else "/kaggle/working/l2"
    print("using", zips[-1], "->", dest)
    subprocess.run(["unzip", "-q", "-o", zips[-1], "-d", dest], check=False)
    os.chdir(dest)
kaggle_src = glob.glob("/kaggle/input/**/src/p1", recursive=True)
if not os.path.isdir("src") and kaggle_src:  # Kaggle dataset made from the zip (Kaggle auto-extracts it)
    root = os.path.dirname(os.path.dirname(kaggle_src[0]))
    print("using Kaggle dataset", root)
    shutil.copytree(root, "/kaggle/working/l2", dirs_exist_ok=True)
    os.chdir("/kaggle/working/l2")
if not os.path.isdir("src") and "YOUR_GITHUB" in REPO_URL:
    raise SystemExit("Project code not found: upload L2_colab.zip to /content (Files pane -> content -> Upload), "
                     "wait for the upload to finish, then run this cell again.")
if not os.path.isdir("src"):
    url = REPO_URL
    try:
        from kaggle_secrets import UserSecretsClient
        url = REPO_URL.replace("https://", "https://" + UserSecretsClient().get_secret("GITHUB_TOKEN") + "@")
    except Exception:
        try:
            from google.colab import userdata
            url = REPO_URL.replace("https://", "https://" + userdata.get("GITHUB_TOKEN") + "@")
        except Exception:
            pass
    subprocess.run(["git", "clone", "--quiet", url, "/tmp/repo"], check=True)
    subprocess.run(["git", "-C", "/tmp/repo", "remote", "set-url", "origin", REPO_URL], check=True)  # drop token
    subprocess.run(["git", "-C", "/tmp/repo", "checkout", "--quiet", GIT_REF], check=True)
    os.chdir("/tmp/repo")
sys.path.insert(0, os.getcwd())
res = subprocess.run([sys.executable, "-m", "src.common.install"], capture_output=True, text=True)
print(res.stdout[-4000:], res.stderr[-4000:])
assert res.returncode == 0, "dependency install failed - see the pip output above"
from src.common.runinfo import library_versions
SEED = 266; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print({"commit": commit, "python": sys.version.split()[0], "torch": torch.__version__,
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "seed": SEED})
print(library_versions())

# Part 1 - Evaluation (golden set, baseline vs final, human evaluation, safety)
`eval/p1_golden_set.csv` was written and committed by the team **before** tuning (C2.3).

In [ ]:
# ===== MODEL CHOICE (edit BACKEND, then Run all) =====
#  "qwen3b" : Qwen2.5-VL-3B, fp16, free (first run scored 0.5)
#  "qwen7b" : Qwen2.5-VL-7B, 4-bit, free, slower (~40 min), better rule-following (scored 0.56)
#  "qwen3"  : free, two models: Qwen2.5-VL-7B (4-bit) describes figures, then Qwen3-14B (4-bit, text-only,
#             stronger reasoning) answers. ~45-60 min on a T4. Set QWEN3_THINKING = True for step-by-step
#             reasoning (better arithmetic, but much slower - only if time allows).
#  "gemini": Google Gemini via the google-genai SDK - FREE key from aistudio.google.com, secret GEMINI_API_KEY
#  "claude" : Claude Opus 5.5 via the Anthropic API - needs an ANTHROPIC_API_KEY secret, costs a few dollars
#  Hosted APIs must be named in the README with their model version (lab rule).
BACKEND = "qwen7b"
QWEN3_MODEL = "Qwen/Qwen3-14B"      # or "Qwen/Qwen3-8B" (faster)
QWEN3_THINKING = False
os.environ["P1_BACKEND"] = "local"   # reset; the branches below override it for hosted backends
os.environ.pop("P1_ANSWER_LLM", None)

def _secret(name):
    if os.environ.get(name):
        return os.environ[name]
    try:
        from google.colab import userdata; return userdata.get(name)
    except Exception:
        try:
            from kaggle_secrets import UserSecretsClient; return UserSecretsClient().get_secret(name)
        except Exception:
            return None

if BACKEND == "gemini":
    os.environ["P1_BACKEND"] = "gemini"; os.environ["P1_VLM"] = os.environ.get("GEMINI_MODEL", "gemini-4-hydrogen")
    os.environ["P1_FINAL_K"] = "10"; os.environ["P1_MAX_IMAGES"] = "4"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "google-genai"], check=True)
    import importlib.metadata as md_; print("google-genai", md_.version("google-genai"))
    key = _secret("GEMINI_API_KEY")
    if not key:
        raise SystemExit("Add a free Gemini API key (aistudio.google.com -> Get API key) as a secret named "
                         "GEMINI_API_KEY (Colab: key icon on the left, enable notebook access), then run again.")
    os.environ["GEMINI_API_KEY"] = key
elif BACKEND == "qwen7b":
    os.environ["P1_VLM"] = "Qwen/Qwen2.5-VL-7B-Instruct"; os.environ["P1_VLM_4BIT"] = "1"
elif BACKEND == "qwen3":
    os.environ["P1_VLM"] = "Qwen/Qwen2.5-VL-7B-Instruct"; os.environ["P1_VLM_4BIT"] = "1"   # figure descriptions
    os.environ["P1_ANSWER_LLM"] = QWEN3_MODEL                                               # answers
    os.environ["P1_LLM_THINKING"] = "1" if QWEN3_THINKING else "0"
    os.environ["P1_FINAL_K"] = "8"
elif BACKEND == "claude":
    os.environ["P1_BACKEND"] = "claude"; os.environ["P1_VLM"] = "claude-opus-5-5"
    os.environ["CLAUDE_EFFORT"] = "high"                              # accuracy over cost
    os.environ["P1_FINAL_K"] = "10"; os.environ["P1_MAX_IMAGES"] = "4"   # more evidence, more chart images
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anthropic"], check=True)
    import importlib.metadata as md_; print("anthropic", md_.version("anthropic"))
    key = _secret("ANTHROPIC_API_KEY")
    if not key:
        raise SystemExit("Add your Anthropic API key as a secret named ANTHROPIC_API_KEY "
                         "(Colab: key icon on the left, enable notebook access), then run again.")
    os.environ["ANTHROPIC_API_KEY"] = key
print("BACKEND =", BACKEND)

In [ ]:
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "outputs"
os.makedirs(OUT, exist_ok=True)
import json, time
import pandas as pd
os.environ.setdefault("P1_WORK_DIR", os.path.join(OUT, "p1_work"))
from src.p1.config import P1Config, baseline_config
from src.p1.retrieve import KnowledgeBase
from src.p1.answer import RAGEngine
from src.p1 import evaluate as E
from src.common.vlm import make_vlm
cfg = P1Config()
vlm = make_vlm(cfg.vlm_model, load_in_4bit=cfg.vlm_4bit)
kb = KnowledgeBase.build(cfg, vlm)
print(json.dumps(kb.summary(), indent=1)[:3000])

In [ ]:
systems = {"baseline (dense only)": baseline_config(cfg), "final (hybrid)": cfg,
           "final + reranker (bonus)": cfg.variant(use_reranker=True)}
results, per_q = [], {}
for name, c in systems.items():
    df = E.run_golden(RAGEngine(c, kb, vlm), "eval/p1_golden_set.csv")
    per_q[name] = df
    df.to_csv(os.path.join(OUT, f"p1_golden_{name.split()[0]}.csv"), index=False)
    results.append(E.summarize(df, name))
pd.DataFrame(results)

In [ ]:
pd.set_option("display.max_colwidth", 100)
per_q["final (hybrid)"][["gid", "category", "gold_status", "pred_status", "correct", "recall@5", "citations", "answer"]]

## Human evaluation (groundedness 1-5, both members) - fill `eval/p1_human_ratings.csv` from the export below

In [ ]:
export = per_q["final (hybrid)"][["gid", "question", "answer", "citations", "figure_references"]]
export.to_csv(os.path.join(OUT, "p1_final_answers_for_rating.csv"), index=False)
print(E.human_eval("eval/p1_human_ratings.csv"))

## Safety (C4): untrusted appendix, false claims and hidden instructions - baseline vs final

In [ ]:
cats = ["injection_appendix_c", "conflict", "revision", "false_premise", "not_found"]
cmp = per_q["baseline (dense only)"][["gid", "category", "question", "pred_status", "correct", "answer"]].merge(
    per_q["final (hybrid)"][["gid", "pred_status", "correct", "answer"]], on="gid", suffixes=("_baseline", "_final"))
cmp[cmp.category.isin(cats)]

## Results table (all numbers above are notebook outputs)

In [ ]:
table = pd.DataFrame(results).set_index("system")
table["human_groundedness_mean"] = None
h = E.human_eval("eval/p1_human_ratings.csv")
if h:
    table.loc["final (hybrid)", "human_groundedness_mean"] = h["mean_overall"]
table